# **MVP — Forecasting de Preço de Ações (Ibovespa)**
Autor: *Preencha com seu nome*  
Curso: *Machine Learning & Analytics — PUC-Rio*

Este notebook implementa um MVP completo de **previsão (forecasting)** do preço de fechamento para um ativo da B3, utilizando o dataset hospedado no GitHub.

## Escopo
- Carga direta do **ZIP no GitHub** (sem dependências extras)
- Preparação e limpeza dos dados
- Engenharia de atributos (lags)
- Divisão temporal (train/val/test) sem vazamento
- **Baseline** de persistência
- Modelagem com **LinearRegression**, **RandomForest**, **GradientBoosting**
- **TimeSeriesSplit** + **GridSearchCV** (hiperparâmetros)
- Métricas: **MAE** e **RMSE**
- Resultados, comparação e próximos passos

> **Boas práticas**: seed fixo, baseline, validação temporal, documentação e reprodutibilidade.


In [ ]:
# 0) Imports e configuração
import io, zipfile, urllib.request, os, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reprodutibilidade
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (11, 5)
sns.set(style='whitegrid')

## 1) Carga dos dados via GitHub (ZIP)

In [ ]:
# URL do arquivo ZIP no GitHub (link confirmado)
DATA_ZIP_URL = "https://github.com/Guisteim/MVP_PUC/raw/91b1898879fbacab8156445f995498aacfc7efa4/bovespa_stocks_csv_ML.zip"

def load_bovespa_zip(url: str) -> pd.DataFrame:
    # Baixa o ZIP em memória sem dependências externas
    with urllib.request.urlopen(url) as resp:
        content = resp.read()
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        # Seleciona o primeiro CSV dentro do ZIP
        csv_names = [n for n in z.namelist() if n.lower().endswith('.csv')]
        if not csv_names:
            raise RuntimeError("Nenhum CSV encontrado dentro do ZIP.")
        target_csv = csv_names[0]
        print("Arquivo CSV encontrado no ZIP:", target_csv)
        with z.open(target_csv) as f:
            # IMPORTANTE: separador é ';'
            df = pd.read_csv(f, sep=';', decimal='.')
    return df

df_raw = load_bovespa_zip(DATA_ZIP_URL)
print("Shape bruto:", df_raw.shape)
df_raw.head()

## 2) Inspeção e limpeza mínima

In [ ]:
print("Colunas:", df_raw.columns.tolist())
print("\nTipos iniciais:")
print(df_raw.dtypes)

# Normaliza nomes de colunas (títulos simples e consistentes)
df = df_raw.copy()
df.columns = [c.strip().title() for c in df.columns]

# Verificamos colunas essenciais
expected = ['Date','Symbol','Close']
missing = [c for c in expected if c not in df.columns]
if missing:
    raise RuntimeError(f"Colunas esperadas ausentes: {missing}")

# Converte datas — dataset possui 'dd/mm/yyyy HH:MM'
df['Date'] = pd.to_datetime(df['Date'].astype(str).str.strip(), format="%d/%m/%Y %H:%M", errors="coerce")

# Converte numéricos que porventura vieram como texto
num_cols = ['Adj Close','Close','High','Low','Open','Volume']
for c in num_cols:
    if c in df.columns:
        # Troca vírgulas por pontos se necessário (segurança) e tenta conversão
        df[c] = (df[c].astype(str).str.replace(',', '.', regex=False)
                 .str.replace(' ', '', regex=False))
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Drop de linhas sem data ou close
df = df.dropna(subset=['Date','Close']).reset_index(drop=True)

# Ordena por papel e data
df = df.sort_values(['Symbol','Date']).reset_index(drop=True)

print("Após limpeza:", df.shape)
df.head()

### 2.1) Visão rápida de qualidade dos dados

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum().sort_values(ascending=False))

print("\nPeríodo (min-max) por Symbol:")
periodo = df.groupby('Symbol')['Date'].agg(['min','max','count']).sort_values('count', ascending=False)
display(periodo.head(10))

## 3) Selecionar um ativo para modelagem

In [ ]:
# Estratégia: escolher o ativo com mais observações (mais denso para série)
sym_counts = df['Symbol'].value_counts()
default_symbol = sym_counts.index[0]
print(f"Ativo padrão escolhido (maior nº de registros): {default_symbol} — {sym_counts.iloc[0]} linhas")

SYMBOL = default_symbol  # altere aqui se quiser fixar, ex.: 'PETR4'
df_sym = df[df['Symbol'] == SYMBOL].copy().reset_index(drop=True)
print("Série selecionada:", SYMBOL, "shape:", df_sym.shape)
df_sym.head()

### 3.1) Série temporal — preço de fechamento

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df_sym['Date'], df_sym['Close'])
plt.title(f"Preço de Fechamento — {SYMBOL}")
plt.xlabel("Data"); plt.ylabel("Close")
plt.tight_layout(); plt.show()

## 4) Engenharia de atributos (lags) e alvo (horizon=1)

In [ ]:
HORIZON = 1  # prever o próximo fechamento
LAGS = 10     # número de defasagens
df_feat = df_sym[['Date','Close']].copy()

# Cria lags de Close
for lag in range(1, LAGS+1):
    df_feat[f'lag_{lag}'] = df_feat['Close'].shift(lag)

# Alvo (close futuro)
df_feat['target'] = df_feat['Close'].shift(-HORIZON)

# Remove NaNs gerados pelos shifts
df_feat = df_feat.dropna().reset_index(drop=True)
print("Shape com features:", df_feat.shape)
df_feat.head()

## 5) Split temporal: treino / validação / teste

In [ ]:
n = len(df_feat)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

train = df_feat.iloc[:train_end].copy()
val   = df_feat.iloc[train_end:val_end].copy()
test  = df_feat.iloc[val_end:].copy()

FEATURES = [f'lag_{i}' for i in range(1, LAGS+1)]
TARGET   = 'target'

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print("Tamanhos —", "train:", len(train), "val:", len(val), "test:", len(test))

## 6) Baseline — Persistência (naive)

In [ ]:
# Prever que o próximo valor será igual ao último observado (lag_1)
y_pred_persist_val  = X_val['lag_1'].values
y_pred_persist_test = X_test['lag_1'].values

def eval_reg(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    print(f"{label} MAE: {mae:,.4f} | RMSE: {rmse:,.4f}")
    return mae, rmse

print("Baseline — validação:")
mae_p_v, rmse_p_v = eval_reg(y_val, y_pred_persist_val, "Val")
print("\nBaseline — teste:")
mae_p_t, rmse_p_t = eval_reg(y_test, y_pred_persist_test, "Test")

## 7) Modelagem — Regressão Linear, Random Forest e Gradient Boosting

In [ ]:
# Helper para treinar e avaliar
def train_and_eval(model, Xtr, ytr, Xv, yv, Xt, yt, label):
    model.fit(Xtr, ytr)
    pred_v = model.predict(Xv)
    pred_t = model.predict(Xt)
    print(f"\n{label} — validação:")
    mae_v, rmse_v = eval_reg(yv, pred_v, "Val")
    print(f"{label} — teste:")
    mae_t, rmse_t = eval_reg(yt, pred_t, "Test")
    return {'model': model, 'mae_val': mae_v, 'rmse_val': rmse_v,
            'mae_test': mae_t, 'rmse_test': rmse_t}

# 7.1) Linear Regression (com padronização)
pipe_lr = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
res_lr = train_and_eval(pipe_lr, X_train, y_train, X_val, y_val, X_test, y_test, "LinearRegression")

In [ ]:
# 7.2) Random Forest — com busca simples de hiperparâmetros via TimeSeriesSplit
tss = TimeSeriesSplit(n_splits=3)
rf = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
grid_rf = {'n_estimators': [100, 300],
           'max_depth': [None, 5, 10]}
gcv_rf = GridSearchCV(rf, grid_rf, cv=tss, scoring='neg_mean_absolute_error', n_jobs=-1)
gcv_rf.fit(X_train, y_train)
best_rf = gcv_rf.best_estimator_
print("RF — melhores parâmetros:", gcv_rf.best_params_)

res_rf = train_and_eval(best_rf, X_train, y_train, X_val, y_val, X_test, y_test, "RandomForest")

In [ ]:
# 7.3) Gradient Boosting — busca simples de hiperparâmetros
gbr = GradientBoostingRegressor(random_state=RANDOM_SEED)
grid_gbr = {'n_estimators': [100, 300],
            'max_depth': [2, 3]}
gcv_gbr = GridSearchCV(gbr, grid_gbr, cv=tss, scoring='neg_mean_absolute_error', n_jobs=-1)
gcv_gbr.fit(X_train, y_train)
best_gbr = gcv_gbr.best_estimator_
print("GBR — melhores parâmetros:", gcv_gbr.best_params_)

res_gbr = train_and_eval(best_gbr, X_train, y_train, X_val, y_val, X_test, y_test, "GradientBoosting")

## 8) Comparação de resultados (val e teste)

In [ ]:
results = {
    'persistence': {'mae_val': mae_p_v, 'rmse_val': rmse_p_v, 'mae_test': mae_p_t, 'rmse_test': rmse_p_t},
    'linear': res_lr,
    'random_forest': res_rf,
    'gbr': res_gbr
}
import pandas as pd
df_res = pd.DataFrame({
    'Model': ['Baseline (Persist.)','Linear','RandomForest','GradientBoosting'],
    'MAE_val': [results['persistence']['mae_val'], results['linear']['mae_val'], results['random_forest']['mae_val'], results['gbr']['mae_val']],
    'RMSE_val':[results['persistence']['rmse_val'],results['linear']['rmse_val'],results['random_forest']['rmse_val'],results['gbr']['rmse_val']],
    'MAE_test':[results['persistence']['mae_test'],results['linear']['mae_test'],results['random_forest']['mae_test'],results['gbr']['mae_test']],
    'RMSE_test':[results['persistence']['rmse_test'],results['linear']['rmse_test'],results['random_forest']['rmse_test'],results['gbr']['rmse_test']]
})
df_res

## 9) Modelo vencedor e avaliação final no teste

In [ ]:
# Escolher pelo menor MAE_val
cands = [('linear', res_lr), ('random_forest', res_rf), ('gbr', res_gbr)]
best_name, best = min(cands, key=lambda x: x[1]['mae_val'])
print("Vencedor na validação:", best_name)

# Refit no conjunto train+val e avaliar no test
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

if best_name == 'linear':
    final_model = pipe_lr
elif best_name == 'random_forest':
    final_model = best_rf
else:
    final_model = best_gbr

final_model.fit(X_trainval, y_trainval)
pred_test = final_model.predict(X_test)
final_mae = mean_absolute_error(y_test, pred_test)
final_rmse = mean_squared_error(y_test, pred_test, squared=False)
print(f"Desempenho final no teste — MAE: {final_mae:,.4f} | RMSE: {final_rmse:,.4f}")

### 9.1) Curva prevista vs. observada (janela de teste)

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(test['Date'], y_test, label='Observado')
plt.plot(test['Date'], pred_test, label='Previsto')
plt.title(f"{SYMBOL} — Observado vs Previsto (janela de teste) — {best_name}")
plt.xlabel("Data"); plt.ylabel("Close")
plt.legend(); plt.tight_layout(); plt.show()

## 10) Conclusões e próximos passos
**Resumo:**
- Splits temporais e baseline garantem comparação justa e sem vazamento.
- Três modelos comparados com **TimeSeriesSplit** e **GridSearch**.
- Seleção pelo menor **MAE na validação**; avaliação final no **teste**.

**Possíveis melhorias:**
- Adicionar *features* de volume, *returns*, médias móveis (SMA/EMA), volatilidade, indicadores técnicos (RSI, MACD).
- Avaliar *multi-horizon* (HORIZON > 1) e janelas de entrada maiores.
- Testar modelos específicos de séries (ARIMA/Prophet) e DL (LSTM/GRU/Temporal CNN).
- Fazer *backtesting* mais rigoroso, com validação em múltiplas janelas deslizantes.
- Considerar *ensembles* (stacking/blending) e previsões probabilísticas.

> **Nota:** Em produção, documente recursos computacionais, tempos de treino, versões de libs e fixe seeds.
